# OPT-09 · IA-2 — RG-ILS-SLP for the pooling problem (runnable demo)

**Course:** Introduction to Optimization Techniques (IA-2) · Dept. of IT, K. J. Somaiya College of Engineering
**Group OPT-09:** Razzan Tambe (15) · Saurabh Tripathi (04) · Vivek Sharma (12)

This notebook runs our code in the browser:

* **Baseline:** our IA-1 re-implementation of Grothey & McKinnon (2023): qq-formulation, filter SLP and random multistart (`qq_model.py`, `slp.py`, unchanged).
* **Proposed:** RG-ILS-SLP (`rg_ils.py`), with diet-LP relaxation, recipe-clustering start, LP repair/polish and a regret-guided iterated local search.

Run the cells from top to bottom (**Runtime → Run all**). The quick demo takes about 2–3 minutes. The full experiment (all 11 instances × 3 seeds × 20 runs) takes about 3.5 h. Its stored results are loaded in Section 4.

## 1. Get the code and install the requirements

In [ ]:
import os, sys
REPO_URL = "https://github.com/razzantambe/OPT-09-IA2-Pooling-SLP.git"   # <- your GitHub repository
if os.path.exists("04_Source_Code"):            # notebook opened inside the repository
    ROOT = os.path.abspath(".")
elif os.path.exists("../04_Source_Code"):
    ROOT = os.path.abspath("..")
else:                                           # Google Colab: clone the repository
    os.system(f"git clone -q {REPO_URL} repo")
    ROOT = os.path.abspath("repo")
CODE = os.path.join(ROOT, "04_Source_Code")
os.chdir(CODE); sys.path.insert(0, CODE)
print("code folder:", CODE)
print(sorted(f for f in os.listdir(CODE) if f.endswith(".py")))

In [ ]:
# SciPy >= 1.15 bundles the HiGHS LP solver used by slp.py and rg_ils.py
import subprocess, scipy
if tuple(int(v) for v in scipy.__version__.split(".")[:2]) < (1, 15):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "scipy>=1.15"], check=True)
    print("SciPy upgraded - please use Runtime -> Restart session, then run all cells again")
import numpy, pandas, matplotlib
print("numpy", numpy.__version__, "| scipy", scipy.__version__, "| pandas", pandas.__version__)

## 2. Look at one instance
`load_instance` and `QQModel` come from IA-1. `diet_relaxation` is new in IA-2. It gives a lower bound that, on af-1, equals Baron's bound (295.93153).

In [ ]:
from qq_model import load_instance, QQModel
from rg_ils import diet_relaxation
DATA = os.path.join(ROOT, "05_Dataset")
inst = load_instance(os.path.join(DATA, "af-1.txt"))
mod = QQModel(inst)
print(f"af-1: {inst.I} raw materials, {inst.M} bins, {inst.P} products, {inst.N} nutrients")
print(f"qq-model: {mod.n} variables, {mod.m_total} constraints")
print(f"diet-LP lower bound: {diet_relaxation(mod)['LB']:.5f}   (Baron LB in the paper: 295.93153, best known 296.44604)")

## 3. Quick demo: baseline vs proposed (10 runs each on af-2 and af-5-fs, about 2–3 min)
This is the same command-line program used for the paper, with fewer runs. With seed 12345 the 10 runs are exactly the first 10 runs of the stored experiment, and only the times depend on the machine.

* **af-2:** the baseline finds no Good run, and the proposed method finds 5 of 10.
* **af-5-fs:** the baseline ends infeasible in 2 of 10 runs, and the proposed method is feasible in all 10.

In [ ]:
import subprocess, pandas as pd
OUT = os.path.join(ROOT, "demo_results")
def run(method, inst, runs=10, seed=12345):
    cmd = [sys.executable, "run_experiments.py", "--method", method, "--instances", inst,
           "--runs", str(runs), "--seed", str(seed), "--outdir", os.path.join(OUT, method)]
    out = subprocess.run(cmd, capture_output=True, text=True).stdout
    print(out.strip().splitlines()[-1])         # summary line: Ti %Feas %Good It ETiGood best f | paper
for inst_name in ["af-2", "af-5-fs"]:
    for method in ["baseline", "proposed"]:
        print(f"--- {method:8s} {inst_name}")
        run(method, inst_name)

In [ ]:
rows = []
for method in ["baseline", "proposed"]:
    for inst_name in ["af-2", "af-5-fs"]:
        d = pd.read_csv(os.path.join(OUT, method, f"runs_{inst_name}.csv"))
        rows.append(dict(instance=inst_name, method=method, runs=len(d), pct_feas=100 * d.feasible.mean(),
                         pct_good=100 * d.good.mean(), Ti=d.time.mean(),
                         best_gap_pct=d.gap_pct[d.feasible == 1].min()))
demo = pd.DataFrame(rows).round(3)
demo

## 4. Full results of the paper (stored)
These are all 11 instances × 3 seeds × 20 runs per method, read from `06_Results`. `analyze_results.py` recomputes them from the per-run CSV files.

In [ ]:
RES = os.path.join(ROOT, "06_Results")
cmp = pd.read_csv(os.path.join(RES, "comparison.csv"))
table = cmp[["instance", "paper_good", "base_feas", "prop_feas", "base_good", "prop_good",
             "base_ETiGood", "prop_ETiGood", "base_best_gap_pct", "prop_best_gap_pct"]].round(2)
table

In [ ]:
stats = pd.read_csv(os.path.join(RES, "stats.csv"))
tests = stats[["instance", "good_base", "good_prop", "p_fisher_good", "feas_base", "feas_prop", "p_mannwhitney_gap"]].round(4)
tests

In [ ]:
from IPython.display import Image, display
for f in ["fig_pct_good.png", "fig_etigood.png", "fig_convergence.png", "fig_ablation.png"]:
    display(Image(os.path.join(RES, "Graphs", f), width=850))

## 5. (Optional) Reproduce everything
Uncomment and run. This takes about 3.5 h on 2 cores, mostly because of af-7b-fs. It overwrites `06_Results`.

In [ ]:
# subprocess.run(["bash", "run_all.sh"], check=True)
# subprocess.run([sys.executable, "analyze_results.py"], check=True)